In [1]:
from pyspark import SparkContext
from pyspark.mllib.recommendation import ALS, Rating
import os

# 1. Initialize Spark Context
sc = SparkContext.getOrCreate()

# 2. Load and Parse the dataset
# Format is User id :: movie id :: ratings :: timestamp
raw_data = sc.textFile("ratings.dat")

def parse_rating(line):
    # The file uses '::' as a separator
    parts = line.split("::")
    # Return a Rating object (user, product, rating)
    # Note: We ignore the timestamp (parts[3])
    return Rating(int(parts[0]), int(parts[1]), float(parts[2]))

ratings = raw_data.map(parse_rating)

# 3. Split the data: 70% Training, 30% Testing
# Using a seed ensures you get the same split every time you run it
training_data, test_data = ratings.randomSplit([0.7, 0.3], seed=42)

# 4. Build the recommendation model (ALS)
# rank: Number of latent factors (features the model discovers)
# iterations: How many times to refine the math
rank = 10
numIterations = 10
model = ALS.train(training_data, rank, numIterations)

# 5. Evaluate the model on test data
# First, create a version of test data without the ratings (user, movie)
test_input = test_data.map(lambda x: (x.user, x.product))

# Predict ratings for the test set
predictions = model.predictAll(test_input).map(lambda r: ((r.user, r.product), r.rating))

# 6. Calculate Mean Squared Error (MSE)
# Join predictions with actual ratings: ((user, movie), (actual, predicted))
rates_and_preds = test_data.map(lambda r: ((r.user, r.product), r.rating)).join(predictions)

# Calculate the average of (actual - predicted)^2
MSE = rates_and_preds.map(lambda r: (r[1][0] - r[1][1])**2).mean()

print(f"Mean Squared Error (MSE) = {MSE}")

# 7. Optional: Save the results to a file if required by your instructor
# Results can be saved to a text file for submission
output_dir = "q2_recommendation_mse"
if os.path.exists(output_dir):
    os.system(f"rm -rf {output_dir}")
sc.parallelize([f"Mean Squared Error: {MSE}"]).saveAsTextFile(output_dir)

Mean Squared Error (MSE) = 0.8285844900441883
